# Piper TTS Training — Iu Mienh Voice

**Setup:** Upload `iu-mienh-tts.tar.gz` as a Kaggle Dataset, enable GPU, turn on Internet.

In [ ]:
# Cell 1: Find our dataset
import os, glob
print('Looking for dataset...')
!find /kaggle/input -maxdepth 4 -name 'metadata.txt' -o -name 'wavs' -type d 2>/dev/null
print('---')
!ls /kaggle/input/ 2>/dev/null
!ls /kaggle/input/*/ 2>/dev/null | head -10

In [ ]:
# Cell 2: Install dependencies
import sys, os
os.environ['DEBIAN_FRONTEND'] = 'noninteractive'
print(f'Python {sys.version}')

!pip install -q pytorch-lightning==1.9.5
!pip install -q onnxruntime librosa tensorboard
!apt-get install -y -qq espeak-ng libespeak-ng-dev
!pip install -q cython

# Clone piper repo
!rm -rf /tmp/piper
!git clone --depth 1 https://github.com/rhasspy/piper.git /tmp/piper

# Build monotonic alignment extension
!cd /tmp/piper/src/python && bash build_monotonic_align.sh

# Add piper_train to Python path (more reliable than pip install -e)
sys.path.insert(0, '/tmp/piper/src/python')

# Install piper_phonemize from source
!pip install -q piper-phonemize 2>/dev/null || echo 'piper-phonemize not on pip, trying source...'
# If pip fails, build from the repo's bundled version
try:
    import piper_phonemize
    print(f'piper_phonemize OK: {piper_phonemize.__file__}')
except ImportError:
    print('Building piper_phonemize from source...')
    !cd /tmp && git clone --depth 1 https://github.com/rhasspy/piper-phonemize.git
    !cd /tmp/piper-phonemize && pip install -q .

# Final check
from piper_train.vits.lightning import VitsModel
print('\n✅ All dependencies ready')

In [ ]:
# Cell 3: Prepare dataset
import sys
sys.path.insert(0, '/tmp/piper/src/python')
from pathlib import Path

# Auto-find metadata.txt wherever Kaggle put it
hits = list(Path('/kaggle/input').glob('**/metadata.txt'))
if not hits:
    raise FileNotFoundError(
        'metadata.txt not found! Check Cell 1 output for actual paths.'
    )

metadata_src = hits[0]
wavs_src = metadata_src.parent / 'wavs'
print(f'Metadata: {metadata_src}')
print(f'Wavs dir: {wavs_src}')
assert wavs_src.is_dir(), f'wavs/ not found at {wavs_src}'

# Setup working dirs
WORK = Path('/kaggle/working/iu-mienh')
DATASET = WORK / 'dataset'
OUTPUT = WORK / 'output'
DATASET.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

# Symlink wavs
wavs_link = DATASET / 'wavs'
if wavs_link.exists() or wavs_link.is_symlink():
    wavs_link.unlink()
wavs_link.symlink_to(wavs_src)

# Write filtered metadata.csv
lines = metadata_src.read_text().strip().split('\n')
valid = [l for l in lines if (wavs_src / l.split('|')[0]).exists()]
meta_csv = DATASET / 'metadata.csv'
meta_csv.write_text('\n'.join(valid) + '\n')

print(f'\n✅ {len(valid)} utterances ready ({len(lines)-len(valid)} skipped)')
for l in valid[:3]:
    p = l.split('|')
    print(f'  {p[0]} | {p[1][:50]}')

In [ ]:
# Cell 4: Preprocess (phonemize as text/codepoints since espeak has no Iu Mienh)
import sys, os
sys.path.insert(0, '/tmp/piper/src/python')
os.environ['PYTHONPATH'] = '/tmp/piper/src/python'
from pathlib import Path

WORK = Path('/kaggle/working/iu-mienh')
DATASET = WORK / 'dataset'
PREPROC = WORK / 'preprocessed'

!PYTHONPATH=/tmp/piper/src/python python3 -m piper_train.preprocess \
    --input-dir {DATASET} \
    --output-dir {PREPROC} \
    --language ium \
    --sample-rate 22050 \
    --dataset-format ljspeech \
    --single-speaker \
    --phoneme-type text \
    --max-workers 2

import json
cfg = json.loads((PREPROC / 'config.json').read_text())
n = sum(1 for _ in open(PREPROC / 'dataset.jsonl'))
print(f'\n✅ Preprocessed: {n} utterances')
print(f'   Phoneme type: {cfg["phoneme_type"]}')
print(f'   Sample rate: {cfg["audio"]["sample_rate"]}')

In [ ]:
# Cell 5: Train
import sys, os
sys.path.insert(0, '/tmp/piper/src/python')
os.environ['PYTHONPATH'] = '/tmp/piper/src/python'
from pathlib import Path

WORK = Path('/kaggle/working/iu-mienh')
PREPROC = WORK / 'preprocessed'
OUTPUT = WORK / 'output'

# Check for existing checkpoint to resume
ckpts = sorted(OUTPUT.glob('**/*.ckpt'))
resume = ''
if ckpts:
    resume = f'--resume_from_checkpoint {ckpts[-1]}'
    print(f'Resuming from: {ckpts[-1]}')
else:
    print('Starting fresh training')

MAX_EPOCHS = 100
BATCH = 16

!PYTHONPATH=/tmp/piper/src/python python3 -m piper_train \
    --dataset-dir {PREPROC} \
    --accelerator gpu \
    --devices 1 \
    --batch-size {BATCH} \
    --validation-split 0.05 \
    --max_epochs {MAX_EPOCHS} \
    --checkpoint-epochs 25 \
    --quality medium \
    --precision 32 \
    --default_root_dir {OUTPUT} \
    {resume}

In [ ]:
# Cell 6: Export to ONNX
import sys, os, shutil
sys.path.insert(0, '/tmp/piper/src/python')
os.environ['PYTHONPATH'] = '/tmp/piper/src/python'
from pathlib import Path

WORK = Path('/kaggle/working/iu-mienh')
OUTPUT = WORK / 'output'
PREPROC = WORK / 'preprocessed'

ckpts = sorted(OUTPUT.glob('**/*.ckpt'))
if not ckpts:
    print('No checkpoints found!')
else:
    ckpt = ckpts[-1]
    onnx_out = Path('/kaggle/working/iu-mienh.onnx')
    print(f'Exporting: {ckpt.name}')

    !PYTHONPATH=/tmp/piper/src/python python3 -m piper_train.export_onnx {ckpt} {onnx_out}

    # Copy config alongside model
    cfg_out = Path('/kaggle/working/iu-mienh.onnx.json')
    shutil.copy2(PREPROC / 'config.json', cfg_out)

    if onnx_out.exists():
        mb = onnx_out.stat().st_size / 1024 / 1024
        print(f'\n✅ Model exported: {mb:.1f} MB')
        print(f'Download from Output tab:')
        print(f'  iu-mienh.onnx')
        print(f'  iu-mienh.onnx.json')

In [ ]:
# Cell 7: Test inference
import sys, os
sys.path.insert(0, '/tmp/piper/src/python')
os.environ['PYTHONPATH'] = '/tmp/piper/src/python'
from pathlib import Path
import IPython.display as ipd

onnx = Path('/kaggle/working/iu-mienh.onnx')
cfg = Path('/kaggle/working/iu-mienh.onnx.json')

if onnx.exists():
    text = 'Tin-Hungh hnamv baamh mienh.'
    out = Path('/kaggle/working/test.wav')
    !echo "{text}" | PYTHONPATH=/tmp/piper/src/python python3 -m piper_train.infer_onnx --model {onnx} --config {cfg} --output-file {out}
    if out.exists():
        print(f'Generated: {text}')
        ipd.display(ipd.Audio(str(out)))
else:
    print('Export model first (Cell 6)')